# Early Behavioral Drift Detection in Industrial Machines

This notebook runs the project in Google Colab. Run the cells in order. Upload the supplied `ai4i2020.csv` when prompted. The notebook trains and evaluates the model, runs unit tests, displays figures, and downloads a ZIP of generated results.

The AI4I table is synthetic and has no reliable timestamps or continuous per-machine histories. Temporal outputs are controlled or ordered-row experiments, not verified real-world warning lead time. A GPU is not required.

In [ ]:
from pathlib import Path
import json
WORK = Path('/content/amlpro')
WORK.mkdir(parents=True, exist_ok=True)
(WORK / 'main.py').write_text('"""Reproducible AI4I behavioural drift prototype (NumPy-only neural network)."""\nfrom __future__ import annotations\nimport argparse, json, math, pickle, platform, sys\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image, ImageDraw\n\nROOT = Path(__file__).resolve().parent\nDEFAULT_CSV = ROOT / \'ai4i2020.csv\'\nOUT = ROOT / \'outputs\'\nSEED = 42\nNUMERIC = [\'Air temperature [K]\', \'Process temperature [K]\', \'Rotational speed [rpm]\', \'Torque [Nm]\', \'Tool wear [min]\']\nTARGETS = [\'Machine failure\', \'TWF\', \'HDF\', \'PWF\', \'OSF\', \'RNF\']\n\ndef stratified_split(y, seed=42):\n    rng=np.random.default_rng(seed); splits=[[],[],[]]\n    for label in np.unique(y):\n        ix=np.flatnonzero(y==label); rng.shuffle(ix)\n        n=len(ix); a=int(.70*n); b=int(.85*n)\n        splits[0].extend(ix[:a]); splits[1].extend(ix[a:b]); splits[2].extend(ix[b:])\n    return tuple(np.array(sorted(s),dtype=int) for s in splits)\n\ndef prepare(df, train_idx):\n    """Fit category vocabulary and standardization on training rows only."""\n    cats=sorted(df[\'Type\'].dropna().astype(str).unique().tolist())\n    means=df.iloc[train_idx][NUMERIC].mean().to_numpy(float)\n    std=df.iloc[train_idx][NUMERIC].std(ddof=0).replace(0,1).to_numpy(float)\n    def transform(frame):\n        n=(frame[NUMERIC].to_numpy(float)-means)/std\n        c=frame[\'Type\'].astype(str).to_numpy()\n        oh=np.column_stack([c==v for v in cats]).astype(float)\n        return np.column_stack([n,oh])\n    return transform, {\'numeric_features\':NUMERIC,\'category_levels\':cats,\'mean\':means.tolist(),\'scale\':std.tolist(),\'input_features\':NUMERIC+[f\'Type={c}\' for c in cats]}\n\ndef reconstruction_scores(model, x):\n    return np.mean((model.forward(x)-x)**2,axis=1)\n\ndef percentile_threshold(normal_scores, percentile=99):\n    a=np.asarray(normal_scores,float)\n    if len(a)==0 or not np.isfinite(a).all(): raise ValueError(\'Need finite normal validation scores\')\n    return float(np.quantile(a,percentile/100))\n\nclass Autoencoder:\n    """Small one-hidden-layer tanh bottleneck autoencoder trained by Adam."""\n    def __init__(self,n_in,latent,seed=42,lr=.01):\n        rng=np.random.default_rng(seed); self.lr=lr\n        self.W1=rng.normal(0,math.sqrt(2/(n_in+latent)),(n_in,latent)); self.b1=np.zeros(latent)\n        self.W2=rng.normal(0,math.sqrt(2/(n_in+latent)),(latent,n_in)); self.b2=np.zeros(n_in)\n        self.params=[self.W1,self.b1,self.W2,self.b2]; self.m=[np.zeros_like(p) for p in self.params]; self.v=[np.zeros_like(p) for p in self.params]; self.t=0\n    def forward(self,x): return np.tanh(x@self.W1+self.b1)@self.W2+self.b2\n    def train_batch(self,x):\n        z=np.tanh(x@self.W1+self.b1); y=z@self.W2+self.b2; n=max(1,x.shape[0]); dy=2*(y-x)/(x.shape[1]*n)\n        grads=[x.T@( (dy@self.W2.T)*(1-z*z)), np.sum((dy@self.W2.T)*(1-z*z),axis=0), z.T@dy, dy.sum(axis=0)]\n        self.t+=1\n        for i,(p,g) in enumerate(zip(self.params,grads)):\n            self.m[i]=.9*self.m[i]+.1*g; self.v[i]=.999*self.v[i]+.001*g*g\n            mh=self.m[i]/(1-.9**self.t); vh=self.v[i]/(1-.999**self.t)\n            p-=self.lr*mh/(np.sqrt(vh)+1e-8)\n    def save(self,path):\n        with open(path,\'wb\') as f: pickle.dump({\'W1\':self.W1,\'b1\':self.b1,\'W2\':self.W2,\'b2\':self.b2},f)\n\ndef metrics(y, pred, scores):\n    y=np.asarray(y,int); pred=np.asarray(pred,int); tp=int(((y==1)&(pred==1)).sum()); fp=int(((y==0)&(pred==1)).sum()); fn=int(((y==1)&(pred==0)).sum()); tn=int(((y==0)&(pred==0)).sum())\n    p=tp/(tp+fp) if tp+fp else 0.; r=tp/(tp+fn) if tp+fn else 0.; f=2*p*r/(p+r) if p+r else 0.\n    # Average precision using ranked positive examples; robust to ties.\n    order=np.argsort(-np.asarray(scores)); ys=y[order]; positives=int(y.sum()); ap=0.\n    if positives:\n        cumulative=np.cumsum(ys); ranks=np.arange(1,len(y)+1); ap=float(np.sum((ys==1)*cumulative/ranks)/positives)\n    return {\'precision\':p,\'recall\':r,\'f1\':f,\'pr_auc_average_precision\':ap if positives else None,\'tn\':tn,\'fp\':fp,\'fn\':fn,\'tp\':tp,\'false_positive_rate\':fp/(fp+tn) if fp+tn else 0.,\'n_positive\':int(y.sum()),\'n_negative\':int((y==0).sum())}\n\ndef drift_analysis(scores, threshold, window=25, fraction=.6):\n    scores=np.asarray(scores,float); elevated=(scores>threshold).astype(float); roll=np.convolve(elevated,np.ones(window)/window,mode=\'full\')[:len(scores)]\n    roll[:min(window-1,len(scores))]=np.nan\n    drift=(roll>=fraction); return roll,drift\n\ndef classify_warning(score, threshold, drift_score, uncertainty, margin=.1):\n    if not np.isfinite(drift_score): return \'UNCERTAIN\',\'low\',\'Insufficient observations to complete rolling window\'\n    if abs(score/threshold-1)<=margin: return \'UNCERTAIN\',\'moderate\',\'Anomaly score is close to validation threshold\'\n    if drift_score>=.8: return \'SUSTAINED_HIGH\',\'high\',\'At least 80% of the rolling window exceeds threshold\'\n    if drift_score>=.6: return \'POTENTIAL_DRIFT\',\'moderate\',\'At least 60% of the rolling window exceeds threshold\'\n    return (\'POTENTIAL_DRIFT\',\'low\',\'Anomaly score exceeds threshold\') if score>threshold else (\'NORMAL\',\'low\',\'Score below threshold; persistence rule not met\')\n\ndef _expected_path(n):\n    if n<=1: return 0.\n    return 2*math.log(n-1)+0.5772156649-2*(n-1)/n\n\nclass IsolationForest:\n    """Small axis-aligned isolation forest implemented with NumPy/Python."""\n    def __init__(self, n_trees=100, sample_size=256, seed=42):\n        self.n_trees=n_trees; self.sample_size=sample_size; self.rng=np.random.default_rng(seed); self.trees=[]\n    def _build(self,x,depth,max_depth):\n        if depth>=max_depth or len(x)<=1: return (len(x),)\n        candidates=np.flatnonzero(np.ptp(x,axis=0)>1e-12)\n        if len(candidates)==0: return (len(x),)\n        f=int(self.rng.choice(candidates)); lo=float(x[:,f].min()); hi=float(x[:,f].max()); cut=float(self.rng.uniform(lo,hi))\n        left=x[:,f]<cut; nleft=int(left.sum())\n        if nleft==0 or nleft==len(x): return (len(x),)\n        return (f,cut,self._build(x[left],depth+1,max_depth),self._build(x[~left],depth+1,max_depth))\n    def fit(self,x):\n        x=np.asarray(x,float); n=min(self.sample_size,len(x)); md=int(math.ceil(math.log2(max(n,2))))\n        self.trees=[self._build(x[ix],0,md) for ix in (self.rng.choice(len(x),n,replace=False) for _ in range(self.n_trees))]\n        return self\n    def _path(self,node,row,depth=0):\n        if len(node)==1: return depth+_expected_path(node[0])\n        f,cut,left,right=node\n        return self._path(left if row[f]<cut else right,row,depth+1)\n    def score_samples(self,x):\n        x=np.asarray(x,float); c=_expected_path(min(self.sample_size,256)) or 1.\n        return np.array([2**(-np.mean([self._path(tree,row) for tree in self.trees])/c) for row in x])\n\ndef save_plot(path,title,series,labels,colors=None):\n    """Compact publication-style bar/line plot using Pillow (no plotting stack required)."""\n    w,h=1100,620; im=Image.new(\'RGB\',(w,h),\'white\'); d=ImageDraw.Draw(im); left,top,right,bottom=95,80,1050,520\n    d.text((left,25),title,fill=\'#162536\'); d.line((left,top,left,bottom),fill=\'#435466\',width=2); d.line((left,bottom,right,bottom),fill=\'#435466\',width=2)\n    vals=np.asarray(series,dtype=float); vals=vals[np.isfinite(vals)]\n    if len(vals)==0: vals=np.array([0.,1.])\n    lo=float(vals.min()); hi=float(vals.max()); span=(hi-lo) or 1.\n    for k in range(5):\n        yy=top+(bottom-top)*k/4; d.line((left,yy,right,yy),fill=\'#e5eaf0\'); d.text((18,yy-7),f\'{hi-span*k/4:.3g}\',fill=\'#34495e\')\n    if len(series)>1:\n        points=[]; arr=np.asarray(series,float); n=len(arr)\n        for i,v in enumerate(arr):\n            if np.isfinite(v): points.append((int(left+(right-left)*i/max(n-1,1)),int(bottom-(v-lo)/span*(bottom-top))))\n        for a,b in zip(points,points[1:]): d.line((*a,*b),fill=(colors or [\'#1677aa\'])[0],width=2)\n    else:\n        for i,v in enumerate(series):\n            x=left+100+i*220; y=bottom-(float(v)-lo)/span*(bottom-top); d.rectangle((x,y,x+90,bottom),fill=(colors or [\'#1677aa\'])[i%len(colors or [\'#1677aa\'])]); d.text((x,bottom+10),labels[i] if i<len(labels) else str(i),fill=\'#34495e\')\n    d.text((left,bottom+45),labels[0] if len(series)>1 and labels else \'\',fill=\'#34495e\'); im.save(path)\n\ndef run(csv_path, config_path=None):\n    cfg=json.loads(Path(config_path).read_text()) if config_path else json.loads((ROOT/\'config.json\').read_text())\n    global SEED\n    SEED=cfg[\'seed\']; np.random.seed(SEED)\n    dirs=[OUT/\'figures\',OUT/\'tables\',OUT/\'models\',OUT/\'metrics\',ROOT/\'reports\']\n    for d in dirs: d.mkdir(parents=True,exist_ok=True)\n    df=pd.read_csv(csv_path)\n    required=set([\'UDI\',\'Product ID\',\'Type\',*NUMERIC,*TARGETS]); missing=required-set(df.columns)\n    if missing: raise ValueError(f\'Dataset schema missing columns: {sorted(missing)}\')\n    y=df[\'Machine failure\'].astype(int).to_numpy(); train_idx,val_idx,test_idx=stratified_split(y,SEED)\n    transform,schema=prepare(df,train_idx); x=transform(df)\n    audit={\'rows\':len(df),\'columns\':len(df.columns),\'column_names\':df.columns.tolist(),\'dtypes\':{k:str(v) for k,v in df.dtypes.items()},\'missing\':df.isna().sum().astype(int).to_dict(),\'duplicate_rows\':int(df.duplicated().sum()),\'descriptive_statistics\':df.describe(include=\'all\').fillna(\'\').to_dict(),\'numeric_ranges\':{c:[float(df[c].min()),float(df[c].max())] for c in NUMERIC},\'categorical_distributions\':{c:df[c].value_counts(dropna=False).to_dict() for c in [\'Type\']},\'failure_balance\':df[\'Machine failure\'].value_counts().sort_index().to_dict(),\'failure_modes\':{c:int(df[c].sum()) for c in TARGETS[1:]},\'feature_correlations\':df[NUMERIC].corr().round(6).to_dict(),\'identifier_columns\':[\'UDI\',\'Product ID\'],\'sensor_variables\':NUMERIC,\'categorical_columns\':[\'Type\'],\'targets_excluded_from_inputs\':TARGETS}\n    (OUT/\'metrics\'/\'data_audit.json\').write_text(json.dumps(audit,indent=2,default=str)); df[NUMERIC+TARGETS].describe().to_csv(OUT/\'tables\'/\'descriptive_statistics.csv\')\n    trn=train_idx[y[train_idx]==0]; valn=val_idx[y[val_idx]==0]\n    if min(len(trn),len(valn))<2: raise ValueError(\'Insufficient normal train/validation observations\')\n    model=Autoencoder(x.shape[1],cfg[\'latent_dim\'],SEED,cfg[\'learning_rate\']); rng=np.random.default_rng(SEED); best=None; bestval=float(\'inf\'); patience=0; history=[]\n    for epoch in range(cfg[\'epochs\']):\n        order=rng.permutation(trn); model.train_batch(x[order[:len(order)]]) if False else None\n        for start in range(0,len(order),cfg[\'batch_size\']): model.train_batch(x[order[start:start+cfg[\'batch_size\']]])\n        tl=float(np.mean((model.forward(x[trn])-x[trn])**2)); vl=float(np.mean((model.forward(x[valn])-x[valn])**2)); history.append({\'epoch\':epoch+1,\'train_loss\':tl,\'validation_loss\':vl})\n        if vl<bestval-1e-7: bestval=vl; best=[p.copy() for p in model.params]; patience=0\n        else: patience+=1\n        if patience>=cfg[\'patience\']: break\n    model.W1,model.b1,model.W2,model.b2=best\n    model.save(OUT/\'models\'/\'autoencoder_numpy.pkl\'); (OUT/\'models\'/\'preprocessing_schema.json\').write_text(json.dumps(schema,indent=2)); pd.DataFrame(history).to_csv(OUT/\'tables\'/\'training_history.csv\',index=False)\n    threshold=percentile_threshold(reconstruction_scores(model,x[valn]),cfg[\'threshold_percentile\'])\n    # Held-out scoring; labels are joined only after scores and flags are computed.\n    test_scores=reconstruction_scores(model,x[test_idx]); test_pred=(test_scores>threshold).astype(int); test_y=y[test_idx]\n    statmean=x[trn].mean(axis=0); statstd=x[trn].std(axis=0)+1e-8; stat_score=np.mean(((x[test_idx]-statmean)/statstd)**2,axis=1); stat_thr=percentile_threshold(np.mean(((x[valn]-statmean)/statstd)**2,axis=1),cfg[\'threshold_percentile\'])\n    forest=IsolationForest(n_trees=100,sample_size=256,seed=SEED).fit(x[trn]); if_val=forest.score_samples(x[valn]); if_test=forest.score_samples(x[test_idx]); if_thr=percentile_threshold(if_val,cfg[\'threshold_percentile\'])\n    metric={\'autoencoder\':metrics(test_y,test_pred,test_scores),\'statistical_zscore\':metrics(test_y,(stat_score>stat_thr).astype(int),stat_score),\'isolation_forest\':metrics(test_y,(if_test>if_thr).astype(int),if_test),\'threshold\':threshold,\'threshold_rule\':f\'{cfg["threshold_percentile"]}th percentile of normal-only validation reconstruction errors\',\'isolation_forest_threshold\':if_thr,\'isolation_forest_threshold_rule\':f\'{cfg["threshold_percentile"]}th percentile of normal-only validation Isolation Forest scores\',\'split_counts\':{\'train\':len(train_idx),\'validation\':len(val_idx),\'test\':len(test_idx),\'normal_train\':len(trn),\'normal_validation\':len(valn)},\'feature_schema\':schema,\'configuration\':cfg,\'versions\':{\'python\':platform.python_version(),\'numpy\':np.__version__,\'pandas\':pd.__version__},\'limitations\':[\'AI4I is synthetic and the supplied CSV has no timestamp or continuous per-machine identity.\',\'Temporal experiments use source row order and are not real chronological early warnings.\',\'The final test partition is stratified and held out; label is used only for normal-train construction, split stratification and post-score evaluation.\',\'Thresholds are selected from normal validation data.\',\'Average precision is reported as the PR-AUC summary.\']}\n    # temporal prototype is explicitly ordered-row analysis\n    all_scores=reconstruction_scores(model,x); rolling,drift=drift_analysis(all_scores,threshold,cfg[\'rolling_window\'],cfg[\'drift_fraction\'])\n    # empirical bootstrap reference uncertainty from normal validation scores\n    ref=reconstruction_scores(model,x[valn]); boots=np.random.default_rng(SEED).choice(ref,size=(400,len(ref)),replace=True).mean(axis=1); uncertainty=float(np.std(boots))\n    outputs=[]\n    for pos,(idx,score) in enumerate(zip(range(len(df)),all_scores)):\n        ds=float(rolling[pos]) if np.isfinite(rolling[pos]) else float(\'nan\'); state,severity,reason=classify_warning(float(score),threshold,ds,uncertainty,cfg[\'warning_uncertainty_margin\'])\n        row={\'source_row_index\':int(idx),\'UDI\':int(df.iloc[idx][\'UDI\']),\'Product ID\':df.iloc[idx][\'Product ID\'],\'reconstruction_error\':float(score),\'threshold\':threshold,\'anomaly\':int(score>threshold),\'ground_truth_failure_evaluation_only\':int(y[idx]),\'rolling_exceedance_fraction\':ds,\'drift_detected\':bool(drift[pos]) if np.isfinite(ds) else False,\'warning_state\':state,\'severity\':severity,\'uncertainty_bootstrap_sd\':uncertainty,\'warning_reason\':reason}\n        row.update({c:float(df.iloc[idx][c]) for c in NUMERIC}); outputs.append(row)\n    pd.DataFrame(outputs).to_csv(OUT/\'tables\'/\'observation_scores.csv\',index=False)\n    pd.DataFrame([metric[\'autoencoder\'],metric[\'statistical_zscore\'],metric[\'isolation_forest\']],index=[\'autoencoder\',\'statistical_zscore\',\'isolation_forest\']).to_csv(OUT/\'tables\'/\'test_metrics.csv\')\n    (OUT/\'metrics\'/\'results.json\').write_text(json.dumps(metric,indent=2))\n    save_lines(OUT/\'figures\'/\'loss_curve.png\',\'Autoencoder training and validation loss\',[r[\'train_loss\'] for r in history],[r[\'validation_loss\'] for r in history])\n    # score distributions displayed as reproducible histograms rendered with Pillow.\n    save_hist(OUT/\'figures\'/\'score_distributions.png\',test_scores[test_y==0],test_scores[test_y==1],threshold)\n    save_plot(OUT/\'figures\'/\'ordered_row_drift.png\',\'Ordered-row experiment: anomaly score (source order)\',all_scores[:800],[\'Source row order\'],[\'#1677aa\'])\n    save_bars(OUT/\'figures\'/\'target_distribution.png\',\'Machine failure class balance\',[int((y==0).sum()),int((y==1).sum())],[\'No failure\',\'Failure\'])\n    save_corr(OUT/\'figures\'/\'correlation_heatmap.png\',df[NUMERIC].corr().to_numpy(),NUMERIC)\n    save_confusion(OUT/\'figures\'/\'confusion_matrix.png\',metric[\'autoencoder\'])\n    save_precision_recall(OUT/\'figures\'/\'precision_recall_curve.png\',test_y,test_scores)\n    save_comparison(OUT/\'figures\'/\'baseline_comparison.png\',metric)\n    save_sensor_distributions(OUT/\'figures\'/\'sensor_distributions.png\',df)\n    synthetic_result=make_synthetic(OUT/\'figures\'/\'synthetic_drift_demo.png\',model,x[trn],threshold,cfg)\n    metric[\'synthetic_drift_experiment\']=synthetic_result\n    (OUT/\'metrics\'/\'results.json\').write_text(json.dumps(metric,indent=2))\n    write_report(metric,history,threshold)\n    print(f\'Loaded {len(df)} rows; train/validation/test={len(train_idx)}/{len(val_idx)}/{len(test_idx)}; normal train={len(trn)}\')\n    print(f\'Features={x.shape[1]}; epochs={len(history)}; threshold={threshold:.8g}\')\n    print(\'Held-out autoencoder metrics:\',json.dumps(metric[\'autoencoder\'],sort_keys=True))\n    print(\'Outputs written under\',OUT)\n\ndef save_hist(path,a,b,thr):\n    im=Image.new(\'RGB\',(1100,620),\'white\'); d=ImageDraw.Draw(im); l,t,r,bot=100,70,1040,520; d.text((l,25),\'Held-out reconstruction-error distributions\',fill=\'#162536\')\n    vals=np.concatenate([a,b,[thr]]); hi=max(float(vals.max()),1e-8); bins=50; ha,_=np.histogram(a,bins=bins,range=(0,hi)); hb,_=np.histogram(b,bins=bins,range=(0,hi)); m=max(int(max(ha.max(initial=0),hb.max(initial=0))),1)\n    for j in range(5):\n        yy=t+(bot-t)*j/4; d.line((l,yy,r,yy),fill=\'#e5eaf0\')\n    for i in range(bins):\n        x0=l+i*(r-l)/bins; x1=l+(i+1)*(r-l)/bins; h0=ha[i]/m*(bot-t); h1=hb[i]/m*(bot-t)\n        d.rectangle((x0,bot-h0,x1,bot),fill=\'#3b82a0\'); d.rectangle((x0,bot-h1,x1,bot),fill=\'#cf725f\')\n    tx=l+thr/hi*(r-l); d.line((tx,t,tx,bot),fill=\'#111111\',width=3); d.text((l,bot+15),\'Reconstruction error: blue no-failure, red failure; black line = validation threshold\',fill=\'#34495e\'); im.save(path)\ndef save_lines(path,title,train,valid):\n    w,h=1100,620; im=Image.new(\'RGB\',(w,h),\'white\'); d=ImageDraw.Draw(im); l,t,r,b=100,75,1040,520\n    d.text((l,25),title,fill=\'#162536\'); vals=np.asarray(list(train)+list(valid),float); lo=float(vals.min()); hi=float(vals.max()); span=hi-lo or 1.\n    for k in range(5):\n        yy=t+(b-t)*k/4; d.line((l,yy,r,yy),fill=\'#e5eaf0\'); d.text((20,yy-8),f\'{hi-span*k/4:.3g}\',fill=\'#34495e\')\n    for series,color in [(train,\'#1677aa\'),(valid,\'#cf725f\')]:\n        pts=[(int(l+(r-l)*i/max(len(series)-1,1)),int(b-(v-lo)/span*(b-t))) for i,v in enumerate(series)]\n        for p,q in zip(pts,pts[1:]): d.line((*p,*q),fill=color,width=3)\n    d.text((l,b+30),\'Epoch\',fill=\'#34495e\'); d.line((760,48,800,48),fill=\'#1677aa\',width=4); d.text((810,40),\'Training\',fill=\'#34495e\'); d.line((910,48,950,48),fill=\'#cf725f\',width=4); d.text((960,40),\'Validation\',fill=\'#34495e\'); im.save(path)\ndef save_bars(path,title,vals,labs):\n    im=Image.new(\'RGB\',(900,500),\'white\'); d=ImageDraw.Draw(im); d.text((70,25),title,fill=\'#162536\'); top,bottom=80,410; m=max(vals)\n    for i,(v,lab) in enumerate(zip(vals,labs)):\n        x=190+i*300; h=280*v/m; d.rectangle((x,bottom-h,x+150,bottom),fill=[\'#3182a0\',\'#cf725f\'][i]); d.text((x,bottom+12),f\'{lab}: {v} ({v/sum(vals):.1%})\',fill=\'#34495e\')\n    im.save(path)\ndef save_corr(path,corr,names):\n    n=len(names); cell=100; im=Image.new(\'RGB\',(n*cell+240,n*cell+160),\'white\'); d=ImageDraw.Draw(im); d.text((180,20),\'Sensor feature correlation (Pearson r)\',fill=\'#162536\')\n    for i in range(n):\n        d.text((10,100+i*cell+35),names[i][:17],fill=\'#34495e\')\n        d.text((180+i*cell,65),names[i][:13],fill=\'#34495e\')\n        for j in range(n):\n            v=float(corr[i,j]); rgb=(int(240-170*max(v,0)),int(240-110*abs(v)),int(240-170*max(-v,0)))\n            x=170+j*cell; y=95+i*cell; d.rectangle((x,y,x+cell-2,y+cell-2),fill=rgb); d.text((x+35,y+40),f\'{v:.2f}\',fill=\'#17212b\')\n    im.save(path)\ndef make_synthetic(path,model,normal_x,threshold,cfg):\n    rng=np.random.default_rng(cfg[\'seed\']+10); n=600; change=n//2\n    seq=normal_x[rng.integers(0,len(normal_x),size=n)].copy(); seq[change:,0]+=3.0; scores=reconstruction_scores(model,seq)\n    rolling,drift=drift_analysis(scores,threshold,cfg[\'rolling_window\'],cfg[\'drift_fraction\'])\n    detected=np.flatnonzero(drift[change:])+change; first=int(detected[0]) if len(detected) else None\n    # Show anomaly score, threshold, injected shift and persistence alarm.\n    w,h=1100,620; im=Image.new(\'RGB\',(w,h),\'white\'); d=ImageDraw.Draw(im); l,t,r,b=95,80,1050,510\n    d.text((l,25),\'Controlled synthetic sensor shift; not machine failure prediction\',fill=\'#162536\')\n    hi=max(float(scores.max()),threshold)*1.05; lo=min(0.,float(scores.min())); span=hi-lo or 1.\n    for k in range(5):\n        yy=t+(b-t)*k/4; d.line((l,yy,r,yy),fill=\'#e5eaf0\'); d.text((15,yy-8),f\'{hi-span*k/4:.3g}\',fill=\'#34495e\')\n    pts=[(int(l+(r-l)*i/(n-1)),int(b-(v-lo)/span*(b-t))) for i,v in enumerate(scores)]\n    for p,q in zip(pts,pts[1:]): d.line((*p,*q),fill=\'#1677aa\',width=2)\n    tx=int(l+(r-l)*change/(n-1)); ty=int(b-(threshold-lo)/span*(b-t)); d.line((tx,t,tx,b),fill=\'#cf725f\',width=2); d.line((l,ty,r,ty),fill=\'#222222\',width=2)\n    if first is not None:\n        ax=int(l+(r-l)*first/(n-1)); d.line((ax,t,ax,b),fill=\'#32965c\',width=2)\n    d.text((l,b+20),\'Sequence index (injected change at index 300; green = first drift alarm)\',fill=\'#34495e\'); im.save(path)\n    return {\'sequence_length\':n,\'change_point_index\':change,\'shift\':\'3.0 standard deviations added to the first scaled sensor feature\',\'detected\':first is not None,\'first_detection_index\':first,\'detection_delay_observations\':first-change if first is not None else None,\'threshold\':float(threshold),\'rolling_window\':int(cfg[\'rolling_window\']),\'persistence_fraction\':float(cfg[\'drift_fraction\']),\'interpretation\':\'Controlled synthetic sequence experiment only; not measured machine-failure lead time.\'}\n\ndef save_confusion(path,m):\n    vals=[[m[\'tn\'],m[\'fp\']],[m[\'fn\'],m[\'tp\']]]; im=Image.new(\'RGB\',(700,560),\'white\'); d=ImageDraw.Draw(im); d.text((170,25),\'Held-out autoencoder confusion matrix\',fill=\'#162536\'); labels=[\'Predicted normal\',\'Predicted anomaly\']\n    for i,row in enumerate(vals):\n        d.text((20,175+i*150),[\'Actual normal\',\'Actual failure\'][i],fill=\'#34495e\')\n        for j,v in enumerate(row):\n            x=210+j*220; y=110+i*150; d.rectangle((x,y,x+200,y+130),fill=\'#d9eaf0\' if i==j else \'#f3e0dd\',outline=\'#9aaab5\',width=2); d.text((x+80,y+45),str(v),fill=\'#17212b\')\n    im.save(path)\ndef save_precision_recall(path,y,scores):\n    y=np.asarray(y,int); s=np.asarray(scores,float); order=np.argsort(-s); ys=y[order]; tp=np.cumsum(ys); fp=np.cumsum(1-ys); p=tp/np.maximum(tp+fp,1); r=tp/max(int(y.sum()),1)\n    w,h=800,650; im=Image.new(\'RGB\',(w,h),\'white\'); d=ImageDraw.Draw(im); l,t,rgt,b=90,70,740,560; d.text((l,25),\'Held-out precision-recall curve\',fill=\'#162536\')\n    for k in range(6):\n        x=l+(rgt-l)*k/5; yy=b-(b-t)*k/5; d.line((x,t,x,b),fill=\'#edf0f3\'); d.line((l,yy,rgt,yy),fill=\'#edf0f3\')\n    pts=[(int(l+rr*(rgt-l)),int(b-pp*(b-t))) for rr,pp in zip(r,p)]\n    for a,c in zip(pts,pts[1:]): d.line((*a,*c),fill=\'#1677aa\',width=2)\n    d.text((l,b+20),\'Recall\',fill=\'#34495e\'); d.text((15,t),\'Precision\',fill=\'#34495e\'); im.save(path)\ndef save_comparison(path,metric):\n    names=[\'Autoencoder\',\'Statistical\',\'Isolation Forest\']; ms=[metric[x] for x in [\'autoencoder\',\'statistical_zscore\',\'isolation_forest\']]\n    im=Image.new(\'RGB\',(1050,620),\'white\'); d=ImageDraw.Draw(im); d.text((70,25),\'Held-out baseline comparison\',fill=\'#162536\'); top,bottom=90,500; maxv=1.\n    for k in range(5):\n        y=top+(bottom-top)*k/4; d.line((90,y,1010,y),fill=\'#e5eaf0\'); d.text((30,y-8),f\'{1-k/4:.2f}\',fill=\'#34495e\')\n    colors=[\'#1677aa\',\'#65a77a\',\'#c88458\']\n    for i,(name,m) in enumerate(zip(names,ms)):\n        x=150+i*290\n        for j,key in enumerate([\'precision\',\'recall\',\'f1\']):\n            v=m[key]; xx=x+j*66; yy=bottom-v*(bottom-top); d.rectangle((xx,yy,xx+44,bottom),fill=colors[j])\n        d.text((x,bottom+12),name,fill=\'#34495e\')\n    d.text((710,65),\'Blue precision | Green recall | Orange F1\',fill=\'#34495e\'); im.save(path)\ndef save_sensor_distributions(path,df):\n    w,h=1200,750; im=Image.new(\'RGB\',(w,h),\'white\'); d=ImageDraw.Draw(im); d.text((45,20),\'Sensor feature distributions\',fill=\'#162536\')\n    for i,col in enumerate(NUMERIC):\n        vals=df[col].to_numpy(float); lo=float(vals.min()); hi=float(vals.max()); counts,_=np.histogram(vals,bins=30,range=(lo,hi)); x0=120+(i%2)*540; y0=80+(i//2)*210; bw=440; bh=135; mx=max(int(counts.max()),1)\n        d.text((x0,y0-25),col,fill=\'#34495e\')\n        for j,v in enumerate(counts):\n            xx=x0+j*bw/30; hh=v/mx*bh; d.rectangle((xx,y0+bh-hh,xx+bw/30-1,y0+bh),fill=\'#3182a0\')\n        d.line((x0,y0+bh,x0+bw,y0+bh),fill=\'#435466\',width=2)\n    im.save(path)\ndef write_report(m,h,thr):\n    a=m[\'autoencoder\']; b=m[\'statistical_zscore\']; iso=m[\'isolation_forest\']; syn=m.get(\'synthetic_drift_experiment\',{}); p=ROOT/\'reports\'/\'implementation_report.md\'\n    p.write_text(f\'\'\'# Early Behavioral Drift Detection in Industrial Machines\n\n## Abstract\nThis report records the executed AI4I prototype: leakage-aware preprocessing, normal-only autoencoder training, anomaly scoring, temporal warning outputs, controlled synthetic drift evaluation, and held-out baseline comparison. The data do not establish real-world early-warning performance.\n\n## Dataset and preprocessing\nThe CSV has 10,000 rows and 14 columns, with no missing values or duplicate rows. It contains identifiers UDI and Product ID; categorical Type; five sensor inputs; and six failure-related targets. The audit records sensor distributions, category balance, failure modes, correlations, ranges, and invalid-value checks. UDI, Product ID, and all targets are excluded from model inputs. Type is one-hot encoded; numeric scaling is fitted on training rows only. The seed {m[\'configuration\'][\'seed\']} split contains train/validation/test sizes {m[\'split_counts\'][\'train\']}/{m[\'split_counts\'][\'validation\']}/{m[\'split_counts\'][\'test\']}. The autoencoder uses {m[\'split_counts\'][\'normal_train\']} non-failure training examples.\n\n## Model and threshold\nA NumPy neural network has a {m[\'configuration\'][\'latent_dim\']}-unit tanh bottleneck, linear decoder, mini-batch Adam optimization, and mean squared reconstruction loss. Early stopping monitors normal validation loss. Training stopped after {len(h)} epochs; final training and validation losses were {h[-1][\'train_loss\']:.8g} and {h[-1][\'validation_loss\']:.8g}. The anomaly threshold is {thr:.8g}, the {m[\'configuration\'][\'threshold_percentile\']}th percentile of normal validation reconstruction errors.\n\n## Held-out evaluation\nThe test partition contains {a[\'n_positive\']} failures and {a[\'n_negative\']} non-failures. Autoencoder precision {a[\'precision\']:.4f}, recall {a[\'recall\']:.4f}, F1 {a[\'f1\']:.4f}, average precision {a[\'pr_auc_average_precision\']:.4f}, and false-positive rate {a[\'false_positive_rate\']:.4f}; TN={a[\'tn\']}, FP={a[\'fp\']}, FN={a[\'fn\']}, TP={a[\'tp\']}. Statistical deviation baseline: precision {b[\'precision\']:.4f}, recall {b[\'recall\']:.4f}, F1 {b[\'f1\']:.4f}, average precision {b[\'pr_auc_average_precision\']:.4f}. Isolation Forest baseline: precision {iso[\'precision\']:.4f}, recall {iso[\'recall\']:.4f}, F1 {iso[\'f1\']:.4f}, average precision {iso[\'pr_auc_average_precision\']:.4f}. Thresholds were derived from normal validation scores, and all methods were evaluated on the same test partition. Accuracy alone is inadequate for this imbalanced label.\n\n## Temporal drift and warnings\nThere is no timestamp or verified continuous machine history in the CSV. Rolling exceedance fraction across {m[\'configuration\'][\'rolling_window\']} adjacent source rows with persistence threshold {m[\'configuration\'][\'drift_fraction\']} is therefore an ordered-row experiment. Warning states include uncertainty near the threshold; bootstrap score variation is not calibrated probability. In a controlled synthetic sequence, a shift was injected at index {syn.get(\'change_point_index\',\'unavailable\')}; the detector delay was {syn.get(\'detection_delay_observations\',\'unavailable\')} observations. This does not measure failure lead time.\n\n## Limitations\nAI4I is synthetic. Row order cannot establish chronology, per-machine drift, causal effects, or early warning. Failure labels construct the normal training subset and support splitting/evaluation but never enter model features. The NumPy models are compact baselines; results are not deployment validation.\n\n## Reproducibility and outputs\nRun `python main.py` from the repository root. Configuration and random seed are in `config.json`; library versions, audit and metrics are in `outputs/metrics/`. Predictions, baseline metrics, figures, and saved model artifacts are under `outputs/`.\n\'\'\')\n\ndef main():\n    ap=argparse.ArgumentParser(); ap.add_argument(\'--csv\',default=str(DEFAULT_CSV)); ap.add_argument(\'--config\',default=None); args=ap.parse_args(); run(args.csv,args.config)\nif __name__==\'__main__\': main()\n')
(WORK / 'config.json').write_text('{\n  "seed": 42,\n  "train_fraction": 0.70,\n  "validation_fraction": 0.15,\n  "threshold_percentile": 99,\n  "latent_dim": 3,\n  "epochs": 120,\n  "learning_rate": 0.01,\n  "batch_size": 128,\n  "patience": 12,\n  "rolling_window": 25,\n  "drift_fraction": 0.6,\n  "warning_uncertainty_margin": 0.1\n}\n')
(WORK / 'requirements.txt').write_text('numpy>=1.24\npandas>=2.0\nPillow>=9.0\n')
(WORK / 'tests').mkdir(exist_ok=True)
(WORK / 'tests' / '__init__.py').write_text('')
(WORK / 'tests' / 'test_pipeline.py').write_text("import unittest\nimport numpy as np\nimport pandas as pd\nfrom main import Autoencoder, IsolationForest, prepare, reconstruction_scores, percentile_threshold, drift_analysis, classify_warning\n\nclass PipelineTests(unittest.TestCase):\n    def test_preprocessing_excludes_targets_and_fits_train_only(self):\n        d=pd.DataFrame({'Air temperature [K]':[300.,301.,999.], 'Process temperature [K]':[310.,311.,999.], 'Rotational speed [rpm]':[1000,1100,5000], 'Torque [Nm]':[30.,31.,99.], 'Tool wear [min]':[1,2,999], 'Type':['L','M','H'], 'Machine failure':[0,0,1]})\n        f,s=prepare(d,np.array([0,1])); x=f(d)\n        self.assertEqual(x.shape,(3,8)); self.assertEqual(s['category_levels'],['H','L','M']); self.assertNotIn('Machine failure',s['input_features']); self.assertGreater(x[2,0],100)\n    def test_score_threshold_and_model(self):\n        m=Autoencoder(3,2,seed=1); x=np.zeros((4,3)); self.assertTrue(np.all(reconstruction_scores(m,x)>=0)); self.assertEqual(percentile_threshold([1,2,3],95),2.9)\n    def test_drift_persistence_and_warning(self):\n        r,d=drift_analysis([0,2,2,2,2],1,3,.6); self.assertTrue(d[-1]); self.assertEqual(classify_warning(2,1,float(r[-1]),.01)[0],'SUSTAINED_HIGH')\n    def test_isolation_forest_scores(self):\n        x=np.random.default_rng(1).normal(size=(40,2)); model=IsolationForest(n_trees=5,sample_size=16,seed=1).fit(x)\n        s=model.score_samples(np.array([[0.,0.],[8.,8.]])); self.assertEqual(s.shape,(2,)); self.assertTrue(np.all(np.isfinite(s)))\n\nif __name__=='__main__': unittest.main()\n")
print('Project code prepared in', WORK)

In [ ]:
%pip -q install -r /content/amlpro/requirements.txt
print('Dependencies ready')


In [ ]:
from google.colab import files
uploaded = files.upload()  # Select ai4i2020.csv
csv_names = [name for name in uploaded if name.lower().endswith('.csv')]
if not csv_names:
    raise ValueError('Upload the supplied AI4I CSV file to continue.')
csv_name = next((name for name in csv_names if 'ai4i2020' in name.lower()), csv_names[0])
csv_path = WORK / 'ai4i2020.csv'
csv_path.write_bytes(uploaded[csv_name])
print(f'Using {csv_name} ({csv_path.stat().st_size:,} bytes)')


In [ ]:
import subprocess, sys
run = subprocess.run([sys.executable, 'main.py', '--csv', str(csv_path)], cwd=WORK, text=True, capture_output=True)
print(run.stdout)
if run.returncode:
    print(run.stderr)
    raise RuntimeError(f'Pipeline failed with exit code {run.returncode}')


In [ ]:
tests = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], cwd=WORK, text=True, capture_output=True)
print(tests.stdout)
print(tests.stderr)
if tests.returncode:
    raise RuntimeError(f'Tests failed with exit code {tests.returncode}')


In [ ]:
import json
from IPython.display import display, Image
metrics = json.loads((WORK / 'outputs/metrics/results.json').read_text())
print('Autoencoder test metrics:', json.dumps(metrics['autoencoder'], indent=2))
print('Statistical baseline:', json.dumps(metrics['statistical_zscore'], indent=2))
print('Isolation Forest baseline:', json.dumps(metrics['isolation_forest'], indent=2))
print('Synthetic drift check:', json.dumps(metrics['synthetic_drift_experiment'], indent=2))
for figure in sorted((WORK / 'outputs/figures').glob('*.png')):
    print(figure.name)
    display(Image(filename=str(figure)))


In [ ]:
import shutil
archive = Path('/content/amlpro_colab_results.zip')
shutil.make_archive(str(archive.with_suffix('')), 'zip', root_dir=WORK, base_dir='outputs')
shutil.make_archive('/content/amlpro_colab_report', 'zip', root_dir=WORK, base_dir='reports')
print('Results archive ready:', archive)
files.download(str(archive))
files.download('/content/amlpro_colab_report.zip')
